# Explore a Tableau workbook with TWB Inspector

This notebook uses the same engine as the TWB Inspector app, so you can explore a workbook
in Python: filter fields, follow the lineage of a calculation, and find everything that depends
on a database column.

**How to use:** set `WORKBOOK` in the first cell to your `.twb` or `.twbx` file and run all cells
(*Run → Run All Cells*). Nothing is uploaded and the workbook is never changed.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make the twb_inspector package importable when the notebook runs from the notebooks/ folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from twb_inspector import analyze_workbook, build_tables, find_workbooks, process_workbook
from twb_inspector.core import STATUS_UNUSED

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)

# 👉 Change this to your own workbook, e.g. r"C:\Users\me\Documents\Sales.twbx"
WORKBOOK = ROOT / "tests" / "samples" / "sample_tables.twb"

lineage = analyze_workbook(str(WORKBOOK))
tables = build_tables(lineage)          # the same tables as the sheets of the Excel audit
print(f"Analysed {lineage.source_file} (Tableau {lineage.tableau_version})")

Analysed sample_tables.twb (Tableau 2024.2.0 (test))


## 1. Summary

In [2]:
pd.Series(lineage.summary(), name="count").to_frame()

,count
Datasources,1
Tables,3
Custom SQL queries,1
Fields (total),8
Raw fields,7
Calculated fields,1
Parameters,0
Groups / sets / bins,0
Worksheets,1
Dashboards,0


## 2. Data sources and tables
Every table each data source reads, with its connection, joins, relationships and Custom SQL.

In [3]:
tables["tables"][["Datasource", "Table", "Physical_Table", "Table_Type", "Connection_Type",
                  "Server", "Database_or_File", "Live_or_Extract", "Columns", "Columns_Used"]]

,Datasource,Table,Physical_Table,Table_Type,Connection_Type,Server,Database_or_File,Live_or_Extract,Columns,Columns_Used
0,Sales model,Customers,[dbo].[Customers],Table,sqlserver,sql01.company.local,DWH,Extract,2,2
1,Sales model,Orders,[dbo].[Orders],Table,sqlserver,sql01.company.local,DWH,Extract,3,2
2,Sales model,Targets,,Custom SQL,sqlserver,sql01.company.local,DWH,Extract,2,2


In [4]:
# Joins, relationships and Custom SQL text
for t in lineage.tables:
    if t.joins or t.relationships or t.sql:
        print(f"■ {t.datasource_caption} / {t.name}")
        for j in t.joins + t.relationships:
            print("   ", j)
        if t.sql:
            print("    SQL:", " ".join(t.sql.split()))

■ Sales model / Orders
    Left join of Orders, Customers on [Orders].[CustomerID] = [Customers].[CustomerID]
    Orders ↔ Targets on [Region] = [region (Targets)]
■ Sales model / Customers
    Left join of Orders, Customers on [Orders].[CustomerID] = [Customers].[CustomerID]
    Orders ↔ Targets on [Region] = [region (Targets)]
■ Sales model / Targets
    Orders ↔ Targets on [Region] = [region (Targets)]
    SQL: SELECT region, target FROM dbo.targets WHERE year = 2026


## 3. All fields
Filter this table like any DataFrame, for example only calculated fields, or only one data source.

In [5]:
fields = tables["fields"]
fields[fields["Type"] == "Calculated"][["Field_Name", "Calc_Kind", "Calculation", "Depends_On",
                                       "Usage_Status", "Worksheets_All"]]

,Field_Name,Calc_Kind,Calculation,Depends_On,Usage_Status,Worksheets_All
0,Amount vs Target,Aggregate,SUM([Amount]) / SUM([target]),"Amount, target",Used on worksheet,Performance


## 4. What can be cleaned up
Unused fields, duplicate calculations and calculations worth a second look.

In [6]:
tables["unused"][["Field_Name", "Type", "Source_Table", "Calculation", "Advice"]]

,Field_Name,Type,Source_Table,Calculation,Advice
0,Order ID,Raw,Orders,,Not used anywhere. Hide it in Tableau to keep extracts smaller.


In [7]:
tables["duplicates"]

,Group,Field_Name,Datasource,Usage_Status,Calculation,Field_ID


In [8]:
tables["flags"][["Field_Name", "Calc_Kind", "Flag", "Usage_Status"]]

,Field_Name,Calc_Kind,Flag,Usage_Status


## 5. Lineage of one field
Everything a field is built from (upstream) and everything that uses it (downstream),
down to worksheets and dashboards.

In [9]:
def find_field(name):
    matches = [f for f in lineage.fields.values() if f.name.lower() == name.lower()]
    if not matches:
        raise ValueError(f"No field called {name!r}")
    return matches[0]


def print_upstream(field, indent=0, seen=None):
    seen = seen or set()
    marker = "  " * indent + ("└─ " if indent else "")
    extra = f"  = {field.formula_friendly}" if field.formula_friendly and field.kind != "Parameter" else ""
    print(f"{marker}{field.name} [{field.kind}]{extra}")
    if field.key in seen:
        return
    seen.add(field.key)
    for k in sorted(field.deps, key=lambda k: lineage.fields[k].name.lower()):
        print_upstream(lineage.fields[k], indent + 1, seen)


def downstream(field):
    out, stack = set(), list(field.used_by)
    while stack:
        k = stack.pop()
        if k not in out:
            out.add(k)
            stack.extend(lineage.fields[k].used_by)
    return [lineage.fields[k].name for k in out]


# 👉 Pick any field name from the table above
FIELD = "Amount vs Target"
f = find_field(FIELD)
print("UPSTREAM (what it is built from)")
print_upstream(f)
print("\nDOWNSTREAM (what uses it):", ", ".join(sorted(downstream(f))) or "nothing")
print("Worksheets:", ", ".join(sorted(f.sheets_all)) or "none")
print("Dashboards:", ", ".join(sorted(f.dashboards_all)) or "none")

UPSTREAM (what it is built from)
Amount vs Target [Calculated]  = SUM([Amount]) / SUM([target])
  └─ Amount [Raw]
  └─ target [Raw]

DOWNSTREAM (what uses it): nothing
Worksheets: Performance
Dashboards: none


## 6. Impact analysis: what breaks if a database column changes?
Give a table and column name as they are in the database. You get every field, calculation,
worksheet and dashboard that depends on it.

In [10]:
def impact(table, column):
    sources = [f for f in lineage.fields.values()
               if f.source_table.lower() == table.lower()
               and (f.source_column or f.name).lower() == column.lower()]
    rows = []
    for src in sources:
        affected = {src.key}
        stack = list(src.used_by)
        while stack:
            k = stack.pop()
            if k not in affected:
                affected.add(k)
                stack.extend(lineage.fields[k].used_by)
        for k in affected:
            fld = lineage.fields[k]
            rows.append({"Field": fld.name, "Type": fld.kind, "Status": fld.status,
                         "Worksheets": ", ".join(sorted(fld.sheets_direct)),
                         "Dashboards": ", ".join(sorted(fld.dashboards_all))})
    return pd.DataFrame(rows).drop_duplicates().sort_values(["Type", "Field"]) if rows else "Not used anywhere"


# 👉 Table and column as in the database
impact("Targets", "target")

,Field,Type,Status,Worksheets,Dashboards
0,Amount vs Target,Calculated,Used on worksheet,Performance,
1,target,Raw,Used via other calculation,,


## 7. Many workbooks at once
Point `FOLDER` at a folder of workbooks to compare them side by side.

In [11]:
# 👉 A folder with .twb / .twbx files
FOLDER = ROOT / "tests" / "samples"

rows = []
for path in find_workbooks(str(FOLDER)):
    s = analyze_workbook(path).summary()
    rows.append({"Workbook": Path(path).name, **{k: s[k] for k in (
        "Tables", "Calculated fields", "Unused calculated fields", "Unused parameters",
        "Duplicate calculation groups", "Worksheets", "Dashboards")}})
pd.DataFrame(rows)

,Workbook,Tables,Calculated fields,Unused calculated fields,Unused parameters,Duplicate calculation groups,Worksheets,Dashboards
0,sample_edge_cases.twb,1,7,2,1,1,2,1
1,sample_tables.twb,3,1,0,0,0,1,0


## 8. Save the Excel audit and the lineage diagram
The same files the app creates.

In [12]:
result = process_workbook(str(WORKBOOK), str(ROOT / "outputs"))
print("Excel:  ", Path(result["excel"]).relative_to(ROOT))
print("Diagram:", Path(result["html"]).relative_to(ROOT))

Excel:   outputs/sample_tables_Calculations_audit.xlsx
Diagram: outputs/sample_tables_lineage_diagram.html
